In [1]:
import pygame
import random
import math
import numpy as np
import matplotlib.pyplot as plt
from class_function import *

pygame 2.6.1 (SDL 2.28.4, Python 3.11.4)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
WIDTH, HEIGHT = 1000, 700
FPS = 60
FOV = 500

BLACK = np.array([10, 10, 15])
WHITE = np.array([255, 255, 255])
FOLDER_COLOR = np.array([100, 200, 255]) # Bleu fluo pour les dossiers
FILE_COLOR = np.array([200, 255, 100])   # Vert fluo pour les fichiers/sous-dossiers
LINE_COLOR = np.array([50, 80, 120])

In [3]:
pygame.init()
screen = pygame.display.set_mode((WIDTH, HEIGHT))
clock = pygame.time.Clock()
running = True

pygame.font.init()
font = pygame.font.SysFont(None, 12)

# --- 1. TÉLÉCHARGEMENT AUTOMATIQUE DU MODÈLE ---
# URL officielle de Google pour le modèle Hand Landmark le plus récent
URL_MODELE = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task"
CHEMIN_MODELE = "hand_landmarker.task"

# Si le fichier n'existe pas dans le dossier, on le télécharge
if not os.path.exists(CHEMIN_MODELE):
    print("Téléchargement du modèle MediaPipe (quelques Mo)... ", end="", flush=True)
    urllib.request.urlretrieve(URL_MODELE, CHEMIN_MODELE)
    print("Terminé !")
else:
    print("Modèle déjà présent localement.")

# --- 2. CONFIGURATION DE LA NOUVELLE API (Tasks) ---
# On indique le chemin du fichier .task fraîchement téléchargé
options_base = python.BaseOptions(model_asset_path=CHEMIN_MODELE)

options = vision.HandLandmarkerOptions(
    base_options=options_base,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_hand_presence_confidence=0.7,
    min_tracking_confidence=0.7
)

# Création du détecteur
detecteur = vision.HandLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)
print("Caméra activée. Appuyez sur 'q' pour quitter.")

chemin_bureau = r"C:\Users\barna\Documents"

bureau_init = generer_depuis_mon_bureau(chemin_bureau, (0, 0, 0), 300, profondeur_max=4)
for folder in bureau_init.folders:
    folder.is_open = True  # Ouvre tous les dossiers par défaut
camera = Camera(0, 0, -FOV)
afficher_popup = False

dico_couleur_main = {'poing': (255, 0, 0), 'baka': (0, 0, 255), 'okay': (255, 255, 0), 'chill': (0, 255, 255), 'rien': (255, 255, 255)}


en_pause_geste = False
temps_action = 0
delai_pause = 2000


while running and cap.isOpened():
    #==============================================
    # 1. PRÉCALCUL DE LA CAMÉRA
    #==============================================

    cam_trig = {
        'cos_x': math.cos(camera.angle_x),
        'sin_x': math.sin(camera.angle_x),
        'cos_y': math.cos(camera.angle_y),
        'sin_y': math.sin(camera.angle_y)
    }
    #===============================================
    #  2 Lecture de la camera dans le bon format
    #===============================================

    succes, frame = cap.read()
    if not succes:
        continue

    # Effet miroir pour que la navigation soit intuitive
    frame = cv2.flip(frame, 1) 
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Convertir en image MediaPipe (Nouveau format API Tasks)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    detection_result = detecteur.detect(mp_image)


    if detection_result.hand_landmarks:
        points = detection_result.hand_landmarks[0]
        okay, baka, poing, chill = deep_three(points), est_corne(points), est_poing_ferme(points), est_chill(points)
        # =========================================================
        # 3. GESTION DES ÉVÉNEMENTS (Actions uniques / Interface)
        # =========================================================
        # Quitter si on appuie sur 'q' sur la fenêtre OpenCV
        if cv2.waitKey(1) & 0xFF == ord('q'):
            running = False

        for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False
        
        if afficher_popup:
            # Fermer le pop_up
            if poing:
                afficher_popup=False
                en_pause_geste = True                      # <--- On active la pause
                temps_action = pygame.time.get_ticks() 
            #Affiche les sous dossiers affiliés
            elif baka:
                if nearest_folder and nearest_folder.children:
                            for child in nearest_folder.children:
                                child.is_open = not child.is_open
                            afficher_popup = False
                            en_pause_geste = True                      # <--- On active la pause
                            temps_action = pygame.time.get_ticks() 
            #Ouvre le dossier selectionné
            elif chill:
                nearest_folder.ouvrir_sur_ordinateur()
                en_pause_geste = True                      # Optionnel: évite d'ouvrir le dossier 15 fois
                temps_action = pygame.time.get_ticks()
        # SI il n'y a pas le pop up
        else:
            # --- GESTION DU CHRONOMÈTRE ---
            if en_pause_geste:
                # Si le temps écoulé dépasse notre délai, on désactive la pause
                if pygame.time.get_ticks() - temps_action > delai_pause:
                    en_pause_geste = False
            if not en_pause_geste:
                if poing:
                    vitesse = zoom(points)
                    dx_avant = -math.sin(camera.angle_y) * math.cos(camera.angle_x)
                    dy_avant = math.sin(camera.angle_x)
                    dz_avant = math.cos(camera.angle_y) * math.cos(camera.angle_x)
                    camera.x += dx_avant * vitesse; camera.y += dy_avant * vitesse; camera.z += dz_avant * vitesse
                elif okay:
                    nearest_folder, _ = bureau_init.find_nearest_folder(camera, cam_trig)
                    if nearest_folder:
                                nom_dossier_cible = nearest_folder.name
                                afficher_popup = True
                
                elif baka:
                    camera.x = 0; camera.y = 0; camera.z = -FOV; camera.angle_x=0; camera.angle_y=0
                
                else:
                    vitesse_rot_y, vitesse_rot_x = tourni(points)
                    camera.angle_y += vitesse_rot_y
                    camera.angle_x += vitesse_rot_x

        
    #     hauteur, largeur = frame.shape[:2]
    #     for i, point in enumerate(points):
    #         x_pixel = int(point.x * largeur)
    #         y_pixel = int(point.y * hauteur)
            
    #         # Poignet (0) et Centre (9) en rouge, le reste en vert
    #         couleur = (0, 0, 255) if i in [0, 9] else (0, 255, 0)
    #         cv2.circle(frame, (x_pixel, y_pixel), 5, couleur, -1)

        
    #     cv2.putText(frame, f"theta :,{tourni(points)} ", (int(largeur*0.3), int(hauteur*0.5)), 
    #                     cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 0, 100), 2)
            
    #     # Vous avez accès aux points avec point.x et point.y (entre 0.0 et 1.0)
    #     # Ex: centre_x = points[9].x
    #     if poing:
    #             cv2.putText(frame, f"one punchh,{zoom(points)} ", (int(largeur*0.6), int(hauteur*0.25)), 
    #                     cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                
            
    #     if baka:
    #         cv2.putText(frame, "corne (tah le basket)", (int(largeur*0.6),int(hauteur*0.25)-10), 
    #                 cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        
    #     if okay:
    #         cv2.putText(frame, "3pnts", (int(largeur*0.6),int(hauteur*0.25)-10), 
    #                 cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    # cv2.imshow("MediaPipe Tasks API - Auto Download", frame)

    # =========================================================
    # 4. RENDU 3D (S'exécute à chaque frame !)
    # =========================================================
    
    pixel_buffer = np.zeros((HEIGHT, WIDTH, 3), dtype=np.float32)
    labels_to_draw = []

    for folder in bureau_init.folders:
        if folder.is_open:
            pixel_buffer, _, _ = folder.draw_folder(pixel_buffer, camera, labels_to_draw, cam_trig)
         


    pixel_buffer = draw_pointer(pixel_buffer)
    pixel_buffer = np.clip(pixel_buffer, 0, 255).astype(np.uint8)
    pygame_buffer = np.swapaxes(pixel_buffer, 0, 1)
    surface = pygame.surfarray.make_surface(pygame_buffer)
    
    screen.blit(surface, (0, 0))
    #cv2.imshow('Debug Vision', frame)

    if detection_result.hand_landmarks:
        hauteur, largeur = frame.shape[:2]

            # --- Paramètres du petit cadre (Minimap) ---
        mini_w, mini_h = 300, 200      # Dimensions du petit cadre en haut à gauche
        marge_x, marge_y = 10, 10      # Décalage par rapport aux bords de l'écran pour faire joli

        # Calcul des ratios de réduction (à faire une seule fois)
        ratio_x = mini_w / largeur
        ratio_y = mini_h / hauteur


        pygame.draw.rect(screen, (30, 30, 30), (marge_x, marge_y, mini_w, mini_h))
        pygame.draw.rect(screen, (255, 255, 255), (marge_x, marge_y, mini_w, mini_h), 2)

        milieu_x = marge_x + (mini_w // 2)
        milieu_y = marge_y + (mini_h // 2)

        # 2. Ligne horizontale (coupe la hauteur en deux)
        # Du bord gauche (marge_x) jusqu'au bord droit (marge_x + mini_w)
        pygame.draw.line(screen, (150, 150, 150), (marge_x, milieu_y), (marge_x + mini_w, milieu_y), 1)

        # 3. Ligne verticale (coupe la largeur en deux)
        # Du bord haut (marge_y) jusqu'au bord bas (marge_y + mini_h)
        pygame.draw.line(screen, (150, 150, 150), (milieu_x, marge_y), (milieu_x, marge_y + mini_h), 1)

        if okay:
             txt_frame = 'okay'
        elif poing:
             txt_frame = 'poing'
        elif chill:
             txt_frame = 'chill'
        elif baka:
             txt_frame = 'baka'
        else:
             txt_frame = 'rien'

        couleur = dico_couleur_main[txt_frame]
        for i, point in enumerate(points):
            x_original = int(point.x * largeur)
            y_original = int(point.y * hauteur)

            # 1. Mise à l'échelle (on réduit la taille)
            x_mini = x_original * ratio_x
            y_mini = y_original * ratio_y
            
            # 2. Translation (on place le point au bon endroit sur l'écran final)
            x_final = int(marge_x + x_mini)
            y_final = int(marge_y + y_mini)
            
            # Poignet (0) et Centre (9) en rouge, le reste en vert
            pygame.draw.circle(screen, couleur, (x_final, y_final), 2)
        
        
    


    # Dessin des labels 3D par dessus les étoiles
    for label, x, y in labels_to_draw:
        text_surface = font.render(label, True, (255, 255, 255))
        screen.blit(text_surface, (x, y))

    
    # =========================================================
    # 5. RENDU DU POPUP (S'affiche par dessus le rendu 3D)
    # =========================================================
    if afficher_popup:
        voile = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
        voile.fill((0, 0, 0, 180)) # Assombrit la 3D derrière
        screen.blit(voile, (0, 0))

        largeur_box, hauteur_box = 700, 150
        x_box = (WIDTH - largeur_box) // 2
        y_box = (HEIGHT - hauteur_box) // 2
        
        pygame.draw.rect(screen, (30, 40, 60), (x_box, y_box, largeur_box, hauteur_box), border_radius=15)
        pygame.draw.rect(screen, (255, 255, 255), (x_box, y_box, largeur_box, hauteur_box), width=3, border_radius=15)

        font_titre = pygame.font.SysFont(None, 40)
        font_sous_titre = pygame.font.SysFont(None, 24)
        
        texte_dossier = font_titre.render(nom_dossier_cible, True, (255, 255, 255))
        
        # On sépare les instructions sur deux lignes pour éviter le débordement
        texte_fermer_1 = font_sous_titre.render("Appuyez sur O pour enlever/ajouter les dossiers connexes", True, (150, 150, 150))
        texte_fermer_2 = font_sous_titre.render("Appuyez sur D pour ouvrir le dossier.", True, (150, 150, 150))
        
        # On ajuste l'espacement vertical (Y) pour faire de la place aux 3 lignes
        screen.blit(texte_dossier, texte_dossier.get_rect(center=(WIDTH // 2, HEIGHT // 2 - 25)))
        screen.blit(texte_fermer_1, texte_fermer_1.get_rect(center=(WIDTH // 2, HEIGHT // 2 + 15)))
        screen.blit(texte_fermer_2, texte_fermer_2.get_rect(center=(WIDTH // 2, HEIGHT // 2 + 40)))
        
    pygame.display.flip()
    clock.tick(FPS)

    

Modèle déjà présent localement.
Caméra activée. Appuyez sur 'q' pour quitter.
[Génération] Lecture de la racine OK : 26 éléments trouvés.
[Erreur sur un sous-dossier ignoré] C:\Users\barna\Documents\Ma musique -> [WinError 5] Accès refusé: 'C:\\Users\\barna\\Documents\\Ma musique'
[Erreur sur un sous-dossier ignoré] C:\Users\barna\Documents\Mes images -> [WinError 5] Accès refusé: 'C:\\Users\\barna\\Documents\\Mes images'
[Erreur sur un sous-dossier ignoré] C:\Users\barna\Documents\Mes vidéos -> [WinError 5] Accès refusé: 'C:\\Users\\barna\\Documents\\Mes vidéos'


KeyboardInterrupt: 